# boIA — detector aéreo + estimación exploratoria de peso

Este cuaderno une el detector YOLO11s de boIA con el regresor MobileNetV3 entrenado con CowDB. Procesa una imagen o video, detecta vacas, estima el peso de cada recorte y guarda una imagen/video anotado y un CSV.

**Límite importante:** CowDB no es un conjunto de imágenes aéreas demostrado. La salida en kg sobre imágenes de dron es una prueba de transferencia, no una medición validada. El ID del tracker solo identifica trayectorias dentro del clip; no identifica vacas entre vuelos.

## 1. Instalar dependencias y montar Drive


In [ ]:
%pip -q install ultralytics

from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')

import os, cv2, torch, numpy as np, pandas as pd
from PIL import Image
from torchvision import models, transforms
from torch import nn
from ultralytics import YOLO

BOIA_RUN = Path('/content/drive/MyDrive/boIA/model_run')  # set your own path
YOLO_PATH = BOIA_RUN / 'weights' / 'best.pt'
REGRESSOR_PATH = Path('/content/cowdb_weight_baseline.pt')
OUT_DIR = Path('/content/boIA/results/integrated_yolo_cowdb_pilot')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, '| YOLO existe:', YOLO_PATH.exists())

## 2. Cargar los dos modelos

Si no encuentra `cowdb_weight_baseline.pt` en `/content`, Colab te pedirá subirlo. Generalo con el notebook de entrenamiento, sujeto a los permisos del dataset. El detector se toma de la carpeta boIA de Drive.


In [ ]:
if not YOLO_PATH.exists():
    raise FileNotFoundError(f'No encuentro el detector: {YOLO_PATH}. Revisá BOIA_RUN.')
if not REGRESSOR_PATH.exists():
    print('Subí cowdb_weight_baseline.pt')
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith('.pt')]
    if not candidates:
        raise FileNotFoundError('No se subió ningún archivo .pt del regresor.')
    REGRESSOR_PATH = Path('/content') / candidates[0]

detector = YOLO(str(YOLO_PATH))
print('YOLO task:', detector.task, '| clases:', detector.names)

# Checkpoint local creado por el cuaderno CowDB. weights_only evita cargar objetos arbitrarios.
try:
    checkpoint = torch.load(REGRESSOR_PATH, map_location='cpu', weights_only=True)
except TypeError:  # require a modern PyTorch version
    raise RuntimeError('Update PyTorch to support weights_only=True')

if checkpoint.get('architecture') != 'mobilenet_v3_small':
    raise ValueError(f"Arquitectura inesperada: {checkpoint.get('architecture')}")
target_mean = float(checkpoint['target_mean'])
target_std = float(checkpoint['target_std'])
regressor = models.mobilenet_v3_small(weights=None)
regressor.classifier[-1] = nn.Linear(regressor.classifier[-1].in_features, 1)
regressor.load_state_dict(checkpoint['state_dict'])
regressor.to(DEVICE).eval()

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print('Regresor cargado; media/std del peso:', target_mean, target_std)

## 3. Subir imagen o video de prueba

En Colab, seleccioná un archivo `.jpg`, `.png`, `.mp4` o `.mov`. Para empezar conviene probar una imagen representativa y revisar visualmente los recortes.


In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('No se subió una imagen o video.')
INPUT_PATH = Path('/content') / next(iter(uploaded))
CONF = 0.25
print('Entrada:', INPUT_PATH)

In [ ]:
def estimate_crop_weight(crop_bgr):
    if crop_bgr is None or crop_bgr.size == 0:
        return float('nan')
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    x = preprocess(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        z = float(regressor(x).squeeze().cpu())
    return z * target_std + target_mean

def infer_frame(frame_bgr, frame_index, persist=True):
    result = detector.track(frame_bgr, persist=persist, conf=CONF,
                            tracker='bytetrack.yaml', verbose=False)[0]
    annotated = frame_bgr.copy()
    rows = []
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return annotated, rows
    coords = boxes.xyxy.cpu().numpy().astype(int)
    confs = boxes.conf.cpu().numpy()
    classes = boxes.cls.cpu().numpy().astype(int)
    track_ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else [None] * len(coords)
    h, w = frame_bgr.shape[:2]
    for i, (xyxy, score, class_id, track_id) in enumerate(zip(coords, confs, classes, track_ids)):
        x1, y1, x2, y2 = xyxy.tolist()
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = frame_bgr[y1:y2, x1:x2]
        kg = estimate_crop_weight(crop)
        label = f'{kg:.0f} kg' + (f' | track {track_id}' if track_id is not None else '')
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (30, 220, 30), 2)
        cv2.putText(annotated, label, (x1, max(20, y1-8)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (30, 220, 30), 2, cv2.LINE_AA)
        rows.append({'frame': frame_index, 'track_id': track_id, 'class_id': int(class_id),
                     'confidence': float(score), 'estimated_weight_kg': float(kg)})
    return annotated, rows

## 4. Ejecutar inferencia y guardar salidas


In [ ]:
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
video_exts = {'.mp4', '.mov', '.avi', '.mkv'}
ext = INPUT_PATH.suffix.lower()
all_rows = []

if ext in image_exts:
    frame = cv2.imread(str(INPUT_PATH))
    if frame is None:
        raise ValueError(f'No pude leer la imagen: {INPUT_PATH}')
    annotated, rows = infer_frame(frame, 0, persist=False)
    all_rows.extend(rows)
    output_media = OUT_DIR / f'{INPUT_PATH.stem}_boia_estimado.jpg'
    cv2.imwrite(str(output_media), annotated)
elif ext in video_exts:
    cap = cv2.VideoCapture(str(INPUT_PATH))
    if not cap.isOpened():
        raise ValueError(f'No pude abrir el video: {INPUT_PATH}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_media = OUT_DIR / f'{INPUT_PATH.stem}_boia_estimado.mp4'
    writer = cv2.VideoWriter(str(output_media), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        annotated, rows = infer_frame(frame, frame_idx, persist=True)
        all_rows.extend(rows)
        writer.write(annotated)
        frame_idx += 1
    cap.release(); writer.release()
else:
    raise ValueError(f'Formato no soportado: {ext}')

predictions = pd.DataFrame(all_rows, columns=['frame','track_id','class_id','confidence','estimated_weight_kg'])
csv_path = OUT_DIR / f'{INPUT_PATH.stem}_predicciones.csv'
predictions.to_csv(csv_path, index=False)
print('Salida visual:', output_media)
print('CSV:', csv_path)
print('Detecciones procesadas:', len(predictions))
display(predictions.head(20))
if ext in image_exts:
    from IPython.display import display as show, Image as IPyImage
    show(IPyImage(filename=str(output_media)))
else:
    from IPython.display import Video
    display(Video(str(output_media), embed=True))

## Lectura correcta del resultado

El detector de vacas y la regresión de peso son módulos distintos. Las métricas de CowDB no se transfieren automáticamente a tomas aéreas. Este cuaderno demuestra que ambos modelos pueden ejecutarse en una misma tubería; para validar peso real hacen falta imágenes y pesajes asociados de los mismos animales, idealmente separados por animal y por sesión/campo. Los `track_id` son temporales dentro del video y no constituyen identificación individual.